# Read one F1 unit as X / y — and save one dataset per sub-model

Loads `data_prep/f1_Xy.parquet` (built by `f1_prep_data.ipynb`), keeps one unit and one horizon,
and returns the feature matrix and target ready for any model. No model is fit here.

Three columns do the filtering: **`unit`**, **`horizon_bd`**, **`split`**.
The features to use are whichever feature columns are non-NaN for that unit.

§1–7 walk through one unit. **§8 does it for every (unit, asset, horizon) and saves each as its own
parquet inside the unit's folder** — one file per sub-model — plus an index of all of them.

## 1 — Load the table

In [21]:
import json, pathlib
import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)

ROOT = pathlib.Path.cwd() / "data_prep"
Xy   = pd.read_parquet(ROOT / "f1_Xy.parquet")
cols = json.load(open(ROOT / "f1_columns.json"))

print(f"{len(Xy):,} rows × {Xy.shape[1]} columns")
print("id columns     :", cols["id"])
print("target columns :", cols["target"])
print("feature columns:", cols["feature"])

150,066 rows × 49 columns
id columns     : ['unit', 'family', 'panel', 'freq', 'asset', 'target_type', 'value_unit', 'asof', 'origin_date', 'origin_idx', 'horizon_bd', 'steps_ahead', 'target_date', 'is_asof', 'split']
target columns : ['anchor', 'target', 'target_change']
feature columns: ['level', 'log_level', 'mom_21', 'mom_63', 'mom_126', 'mom_252', 'z_252', 'pos_252', 'rv_21', 'rv_63', 'rv_252', 'ewma_vol', 'vol_ratio', 'skew_252', 'last_step', 'ctx_mkt_mom_63', 'ctx_mkt_rv_63', 'ctx_mom_mom_63', 'ctx_usd_mom_63', 'ctx_usd_rv_63', 'ctx_slope_10_2', 'ctx_slope_5_2', 'ctx_curv_2_5_10', 'ctx_ust2y', 'ctx_ust10y', 'ctx_slope_mom_63', 'ctx_cpi_yoy', 'ctx_core_yoy', 'ctx_unrate', 'ctx_unrate_chg_3', 'ctx_nfp_3m']


## 2 — Pick a unit and a horizon

In [22]:
UNIT    = "t2-F1-hawkish-cut-2024"
HORIZON = 126          # this unit has [126, 189]; fit one model per horizon

available = Xy[Xy.unit == UNIT].groupby(["asset", "horizon_bd"]).size().rename("rows")
available

asset   horizon_bd
UST_2Y  126           6120
        189           6057
Name: rows, dtype: int64

## 3 — Filter: unit → horizon → split

In [23]:
g = Xy[(Xy.unit == UNIT) & (Xy.horizon_bd == HORIZON)]
assert g.asset.nunique() == 1, f"add an asset filter: {sorted(g.asset.unique())}"   # only pause-2006 has two

train   = g[g.split == "train"]      # target known
predict = g[g.split == "predict"]    # the single as-of row

print(f"unit={UNIT}  asset={g.asset.iloc[0]}  target_type={g.target_type.iloc[0]}  unit={g.value_unit.iloc[0]}")
print(f"as-of={predict.origin_date.iloc[0].date()}  anchor={predict.anchor.iloc[0]}")
print(f"train rows={len(train):,}  predict rows={len(predict)}")

unit=t2-F1-hawkish-cut-2024  asset=UST_2Y  target_type=level  unit=percent_per_annum
as-of=2024-12-18  anchor=4.35
train rows=6,119  predict rows=1


## 4 — Which features apply to this unit

Every feature column that is non-NaN somewhere in the unit's training rows. That is the 15 per-series
features plus the `ctx_*` block of the unit's panel (curve features for rates, USD features for FX, …).
Columns that are all-NaN belong to other panels and are dropped.

In [24]:
FEATURES = [c for c in cols["feature"] if train[c].notna().any()]
dropped  = [c for c in cols["feature"] if c not in FEATURES]
print(f"{len(FEATURES)} features used  :", FEATURES)
print(f"{len(dropped)} dropped (other panels):", dropped)

21 features used  : ['level', 'log_level', 'mom_21', 'mom_63', 'mom_126', 'mom_252', 'z_252', 'pos_252', 'rv_21', 'rv_63', 'rv_252', 'ewma_vol', 'vol_ratio', 'skew_252', 'last_step', 'ctx_slope_10_2', 'ctx_slope_5_2', 'ctx_curv_2_5_10', 'ctx_ust2y', 'ctx_ust10y', 'ctx_slope_mom_63']
10 dropped (other panels): ['ctx_mkt_mom_63', 'ctx_mkt_rv_63', 'ctx_mom_mom_63', 'ctx_usd_mom_63', 'ctx_usd_rv_63', 'ctx_cpi_yoy', 'ctx_core_yoy', 'ctx_unrate', 'ctx_unrate_chg_3', 'ctx_nfp_3m']


## 5 — X and y

Target is **`target_change`** ($x_{t+h}-x_t$); add `anchor` back after predicting. The first ~252
rows have NaN rolling windows — drop them (or keep them for a NaN-tolerant model).

In [25]:
train_ok = train.dropna(subset=FEATURES)

X_train = train_ok[FEATURES]
y_train = train_ok["target_change"]
X_predict = predict[FEATURES]
anchor = float(predict.anchor.iloc[0])

print(f"X_train {X_train.shape}   y_train {y_train.shape}   X_predict {X_predict.shape}   (dropped {len(train)-len(train_ok)} warm-up rows)")
print(f"forecast = anchor + predicted change = {anchor} + model(X_predict)")

X_train (5867, 21)   y_train (5867,)   X_predict (1, 21)   (dropped 252 warm-up rows)
forecast = anchor + predicted change = 4.35 + model(X_predict)


In [26]:
print("y_train (target_change):"); display(y_train.describe().round(3).to_frame().T)
print("X_train (first rows):");    display(X_train.head().round(4))
print("X_predict — the as-of feature vector:")
X_predict.T.rename(columns={X_predict.index[0]: str(predict.origin_date.iloc[0].date())}).round(4)

y_train (target_change):


,count,mean,std,min,25%,50%,75%,max
target_change,5867.0,-0.006,0.753,-2.79,-0.33,0.03,0.36,2.79


X_train (first rows):


,level,log_level,mom_21,mom_63,mom_126,mom_252,z_252,pos_252,rv_21,rv_63,rv_252,ewma_vol,vol_ratio,skew_252,last_step,ctx_slope_10_2,ctx_slope_5_2,ctx_curv_2_5_10,ctx_ust2y,ctx_ust10y,ctx_slope_mom_63
79330,4.92,1.5933,-0.70,-1.06,-1.39,-1.46,-3.2827,0.0243,0.0727,0.0531,0.0491,0.0735,1.4801,-0.4455,0.05,0.22,0.02,-0.18,4.92,5.14,0.37
79331,4.77,1.5623,-0.82,-1.23,-1.52,-1.53,-3.5456,0.0000,0.0770,0.0554,0.0497,0.0768,1.5485,-0.4968,-0.15,0.26,0.05,-0.16,4.77,5.03,0.39
79332,4.56,1.5173,-0.93,-1.47,-1.78,-1.82,-3.9091,0.0000,0.0847,0.0600,0.0511,0.0853,1.6589,-0.6657,-0.21,0.37,0.10,-0.17,4.56,4.93,0.50
79333,4.54,1.5129,-0.88,-1.48,-1.75,-1.81,-3.8227,0.0000,0.0847,0.0600,0.0510,0.0830,1.6583,-0.6683,-0.02,0.40,0.11,-0.18,4.54,4.94,0.55
79334,4.64,1.5347,-0.81,-1.34,-1.67,-1.67,-3.4931,0.0418,0.0889,0.0619,0.0515,0.0879,1.7278,-0.6481,0.10,0.34,0.09,-0.16,4.64,4.98,0.50


X_predict — the as-of feature vector:


,2024-12-18
level,4.3500
log_level,1.4702
mom_21,0.0600
mom_63,0.7600
mom_126,-0.4000
mom_252,-0.0200
z_252,-0.0602
pos_252,0.5548
rv_21,0.0531
rv_63,0.0552


## 6 — Keep the dates alongside (for time-blocked validation later)

In [27]:
meta_train = train_ok[["origin_date", "target_date", "anchor", "target"]]
meta_train.tail(3)

,origin_date,target_date,anchor,target
85194,2024-06-13,2024-12-16,4.68,4.25
85195,2024-06-14,2024-12-17,4.67,4.25
85196,2024-06-17,2024-12-18,4.75,4.35


## 7 — Same thing as one function

Use this from any other notebook: `from load_unit_xy import load_unit_xy` (same code, saved in
[load_unit_xy.py](load_unit_xy.py)).

In [28]:
def load_unit_xy(Xy: pd.DataFrame, cols: dict, unit: str, horizon: int, asset: str | None = None, dropna: bool = True) -> dict:
    g = Xy[(Xy.unit == unit) & (Xy.horizon_bd == horizon)]
    if g.empty:
        raise ValueError(f"no rows for unit={unit!r} horizon={horizon}; available: {sorted(Xy.loc[Xy.unit == unit, 'horizon_bd'].unique())}")
    if asset is not None:
        g = g[g.asset == asset]
    elif g.asset.nunique() > 1:
        raise ValueError(f"{unit} has assets {sorted(g.asset.unique())}; pass asset=")
    train, predict = g[g.split == "train"], g[g.split == "predict"]
    features = [c for c in cols["feature"] if train[c].notna().any()]
    if dropna:
        train = train.dropna(subset=features)
    return {"unit": unit, "asset": predict.asset.iloc[0], "horizon_bd": horizon,
            "target_type": predict.target_type.iloc[0], "asof": predict.origin_date.iloc[0],
            "anchor": float(predict.anchor.iloc[0]), "features": features,
            "X_train": train[features], "y_train": train["target_change"],
            "dates_train": train["origin_date"], "X_predict": predict[features]}

# every F1 unit at its first horizon, to show the feature count varies by panel
rows = []
for (u, h), _ in Xy.groupby(["unit", "horizon_bd"]):
    if h != Xy.loc[Xy.unit == u, "horizon_bd"].min(): continue
    for a in sorted(Xy.loc[Xy.unit == u, "asset"].unique()):
        d = load_unit_xy(Xy, cols, u, h, asset=a)
        rows.append({"unit": u, "asset": a, "horizon": h, "target_type": d["target_type"],
                     "n_train": len(d["y_train"]), "n_features": len(d["features"]), "anchor": d["anchor"]})
pd.DataFrame(rows)

,unit,asset,horizon,target_type,n_train,n_features,anchor
0,t2-F1-ai-mom-2024,MOM,127,log_return,5763,16,0.0000
1,t2-F1-aud-on-hold-2016,AUD,126,level,3854,17,0.7659
2,t2-F1-cad-boc-2017,CAD,126,level,4026,17,1.2768
3,t2-F1-chf-highly-valued-2021,CHF,126,level,5006,17,0.9173
4,t2-F1-conflicting-texts-2024,UST_10Y,126,level,5738,21,4.3100
5,t2-F1-considerable-period-2003,UST_10Y,126,level,525,21,4.3700
6,t2-F1-conundrum-2005,UST_10Y,126,level,905,21,4.2700
7,t2-F1-cpi-glidepath-2023,CPI_ALL,140,level,261,18,303.3340
8,t2-F1-dkk-peg-2019,DKK,129,level,4428,17,6.5591
9,t2-F1-eur-range-2017,EUR,126,level,4066,17,1.2028


## 8 — One parquet per sub-model, saved into the unit folder

A sub-model is one **(unit, asset, horizon)**. For each, keep the id/target columns and only the
features that apply, and write

```
units/<unit>/xy/<asset>__h<horizon>.parquet
```

The file goes in an `xy/` **sub**folder on purpose: the reference CLI (`cli.py:48`) reads every
`*.parquet` sitting directly in a unit folder as a panel, and would choke on ours. The subfolder is
invisible to it.

In [29]:
KEEP_ID = ["unit", "panel", "freq", "asset", "target_type", "value_unit", "asof",
           "origin_date", "origin_idx", "horizon_bd", "steps_ahead", "target_date", "split"]
KEEP_TARGET = ["anchor", "target", "target_change"]

def submodel_frame(Xy: pd.DataFrame, cols: dict, unit: str, asset: str, horizon: int) -> tuple[pd.DataFrame, list[str]]:
    g = Xy[(Xy.unit == unit) & (Xy.asset == asset) & (Xy.horizon_bd == horizon)]
    train = g[g.split == "train"]
    features = [c for c in cols["feature"] if train[c].notna().any()]      # this unit's applicable features
    return g[KEEP_ID + KEEP_TARGET + features].reset_index(drop=True), features

index = []
for (unit, asset, horizon), _ in Xy.groupby(["unit", "asset", "horizon_bd"]):
    df, features = submodel_frame(Xy, cols, unit, asset, horizon)
    out_dir = pathlib.Path.cwd() / "units" / unit / "xy"
    out_dir.mkdir(exist_ok=True)
    path = out_dir / f"{asset}__h{horizon}.parquet"
    df.to_parquet(path, index=False)
    index.append({"unit": unit, "asset": asset, "horizon_bd": horizon, "target_type": df["target_type"].iloc[0],
                  "panel": df["panel"].iloc[0], "freq": df["freq"].iloc[0], "asof": df["asof"].iloc[0].date(),
                  "n_train": int((df.split == "train").sum()), "n_predict": int((df.split == "predict").sum()),
                  "n_features": len(features), "features": ",".join(features),
                  "path": str(path.relative_to(pathlib.Path.cwd()))})

index = pd.DataFrame(index)
index.to_csv(ROOT / "f1_submodels.csv", index=False)
print(f"{len(index)} sub-model files written; index at data_prep/f1_submodels.csv")
index.drop(columns="features")

43 sub-model files written; index at data_prep/f1_submodels.csv


,unit,asset,horizon_bd,target_type,panel,freq,asof,n_train,n_predict,n_features,path
0,t2-F1-ai-mom-2024,MOM,127,log_return,factors_daily,daily,2024-05-31,6015,1,16,units/t2-F1-ai-mom-2024/xy/MOM__h127.parquet
1,t2-F1-aud-on-hold-2016,AUD,126,level,g10_fx_daily,daily,2016-11-01,4106,1,17,units/t2-F1-aud-on-hold-2016/xy/AUD__h126.parquet
2,t2-F1-aud-on-hold-2016,AUD,189,level,g10_fx_daily,daily,2016-11-01,4043,1,17,units/t2-F1-aud-on-hold-2016/xy/AUD__h189.parquet
3,t2-F1-cad-boc-2017,CAD,126,level,g10_fx_daily,daily,2017-07-12,4278,1,17,units/t2-F1-cad-boc-2017/xy/CAD__h126.parquet
4,t2-F1-cad-boc-2017,CAD,189,level,g10_fx_daily,daily,2017-07-12,4215,1,17,units/t2-F1-cad-boc-2017/xy/CAD__h189.parquet
5,t2-F1-chf-highly-valued-2021,CHF,126,level,g10_fx_daily,daily,2021-06-17,5258,1,17,units/t2-F1-chf-highly-valued-2021/xy/CHF__h12...
6,t2-F1-chf-highly-valued-2021,CHF,189,level,g10_fx_daily,daily,2021-06-17,5195,1,17,units/t2-F1-chf-highly-valued-2021/xy/CHF__h18...
7,t2-F1-conflicting-texts-2024,UST_10Y,126,level,rates_daily,daily,2024-06-12,5990,1,21,units/t2-F1-conflicting-texts-2024/xy/UST_10Y_...
8,t2-F1-conflicting-texts-2024,UST_10Y,189,level,rates_daily,daily,2024-06-12,5927,1,21,units/t2-F1-conflicting-texts-2024/xy/UST_10Y_...
9,t2-F1-considerable-period-2003,UST_10Y,126,level,rates_daily,daily,2003-08-12,777,1,21,units/t2-F1-considerable-period-2003/xy/UST_10...


### 8.0 — Which file uses which features

One row per sub-model file, one column per feature, `True` where that feature is in the file.
Saved as `data_prep/f1_submodel_features.csv` (and `.parquet`). Read it to get a file's feature list
without opening the file: `feat_map.loc[path]` → the boolean row → `.index[row]`.

In [30]:
feat_map = pd.DataFrame(
    [{**{"path": r["path"], "unit": r["unit"], "asset": r["asset"], "horizon_bd": r["horizon_bd"], "panel": r["panel"]},
      **{f: (f in r["features"].split(",")) for f in cols["feature"]}} for _, r in index.iterrows()]
).set_index("path")
feat_map.to_csv(ROOT / "f1_submodel_features.csv")
feat_map.to_parquet(ROOT / "f1_submodel_features.parquet")

print(f"{feat_map.shape[0]} files × {len(cols['feature'])} features   "
      f"(per-file feature counts: {sorted(feat_map[cols['feature']].sum(axis=1).unique())})")

# how the feature set differs by panel: which features are on/off per panel
by_panel = feat_map.groupby("panel")[cols["feature"]].all().T.replace({True: "✓", False: ""})
by_panel

43 files × 31 features   (per-file feature counts: [np.int64(16), np.int64(17), np.int64(18), np.int64(21)])


panel,factors_daily,g10_fx_daily,macro_monthly,rates_daily
level,,✓,✓,✓
log_level,,✓,✓,✓
mom_21,✓,✓,✓,✓
mom_63,✓,✓,✓,✓
mom_126,✓,✓,✓,✓
mom_252,✓,✓,✓,✓
z_252,✓,✓,✓,✓
pos_252,✓,✓,✓,✓
rv_21,✓,✓,,✓
rv_63,✓,✓,✓,✓


In [31]:
# example: the feature list for one file, straight from the map
path = "units/t2-F1-pause-2006/xy/UST_10Y__h189.parquet"
row = feat_map.loc[path, cols["feature"]]
print(path); print(list(row.index[row.astype(bool)]))

units/t2-F1-pause-2006/xy/UST_10Y__h189.parquet
['level', 'log_level', 'mom_21', 'mom_63', 'mom_126', 'mom_252', 'z_252', 'pos_252', 'rv_21', 'rv_63', 'rv_252', 'ewma_vol', 'vol_ratio', 'skew_252', 'last_step', 'ctx_slope_10_2', 'ctx_slope_5_2', 'ctx_curv_2_5_10', 'ctx_ust2y', 'ctx_ust10y', 'ctx_slope_mom_63']


### 8.1 — What one unit's folder looks like now

The original fixtures are untouched; the new `xy/` subfolder holds one file per (asset, horizon).

In [32]:
import subprocess
print(subprocess.run(["ls", "-R", "units/t2-F1-pause-2006"], capture_output=True, text=True).stdout)

card.toml
forecast_card.md
forecast_spec.json
manifest.json
rates_daily.parquet
text
xy

units/t2-F1-pause-2006/text:
corpus_index.json
fomc_statement_20060510.txt
fomc_statement_20060629.txt
fomc_statement_20060808.txt

units/t2-F1-pause-2006/xy:
UST_10Y__h126.parquet
UST_10Y__h189.parquet
UST_2Y__h126.parquet
UST_2Y__h189.parquet



### 8.2 — Read one sub-model back

This is the whole loading step for a model: one file, `split` to separate train from predict,
`anchor` to carry into the forecast, every other non-id column is a feature.

In [33]:
sm = pd.read_parquet("units/t2-F1-pause-2006/xy/UST_10Y__h189.parquet")
features = [c for c in sm.columns if c not in KEEP_ID + KEEP_TARGET]

train, predict = sm[sm.split == "train"].dropna(subset=features), sm[sm.split == "predict"]
X_train, y_train = train[features], train["target_change"]
X_predict, anchor = predict[features], float(predict.anchor.iloc[0])

print(f"{sm["unit"].iloc[0]}  {sm["asset"].iloc[0]}  h={sm["horizon_bd"].iloc[0]}  as-of={predict.origin_date.iloc[0].date()}")
print(f"X_train {X_train.shape}  y_train {y_train.shape}  X_predict {X_predict.shape}  anchor={anchor}")
print("features:", features)

t2-F1-pause-2006  UST_10Y  h=189  as-of=2006-08-08
X_train (1209, 21)  y_train (1209,)  X_predict (1, 21)  anchor=4.93
features: ['level', 'log_level', 'mom_21', 'mom_63', 'mom_126', 'mom_252', 'z_252', 'pos_252', 'rv_21', 'rv_63', 'rv_252', 'ewma_vol', 'vol_ratio', 'skew_252', 'last_step', 'ctx_slope_10_2', 'ctx_slope_5_2', 'ctx_curv_2_5_10', 'ctx_ust2y', 'ctx_ust10y', 'ctx_slope_mom_63']


### 8.3 — Loop template for training

When you get to modelling, this is the iteration: one fit per row of the index, one forecast per file.
Two assets in `pause-2006` and two horizons in most units are just more rows in the same loop.

In [34]:
index = pd.read_csv(ROOT / "f1_submodels.csv")
for _, r in index.head(5).iterrows():            # .head(5) just to keep the output short
    sm = pd.read_parquet(r.path)
    features = r.features.split(",")
    train, predict = sm[sm.split == "train"].dropna(subset=features), sm[sm.split == "predict"]
    # model = fit(train[features], train["target_change"])
    # change_draws = model.sample(predict[features], n=2000)
    # value_draws = predict.anchor.iloc[0] + change_draws      -> forecast.parquet rows for (asset, horizon)
    print(f"{r.unit:<32} {r.asset:<8} h={r.horizon_bd:<4} train={len(train):>5}  features={len(features)}  anchor={predict.anchor.iloc[0]}")

t2-F1-ai-mom-2024                MOM      h=127  train= 5763  features=16  anchor=0.0
t2-F1-aud-on-hold-2016           AUD      h=126  train= 3854  features=17  anchor=0.7659
t2-F1-aud-on-hold-2016           AUD      h=189  train= 3791  features=17  anchor=0.7659
t2-F1-cad-boc-2017               CAD      h=126  train= 4026  features=17  anchor=1.2768
t2-F1-cad-boc-2017               CAD      h=189  train= 3963  features=17  anchor=1.2768
